In [ ]:
!pip install -q --upgrade pip

In [ ]:
#nvidia-smi

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
!pip install fsspec==2025.3.0 --force-reinstall
!pip install --upgrade datasets

In [ ]:
!pip install datasets
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

In [ ]:
!pip install evaluate

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
from evaluate import load as load_metric
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
nltk.download('punkt')

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from transformers import AutoTokenizer

In [ ]:
model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

In [ ]:
#Downloading and Unzipping data
!wget https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
!unzip summarizer-data.zip

In [ ]:
dataset = load_from_disk('samsum_dataset')
dataset

In [ ]:
def convert_examples_to_features(example_batch):
  input_encoding = tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)

  with tokenizer.as_target_tokenizer():
    target_encodings = tokenizer(example_batch['summary'], max_length=140, truncation=True)

  return{
      'input_ids' : input_encoding['input_ids'],
      'attention_mask' : input_encoding['attention_mask'],
      'labels': target_encodings['input_ids']
  }

In [ ]:
dataset_samsum = dataset.map(convert_examples_to_features, batched = True)

In [ ]:
dataset_samsum['validation']

In [ ]:
## Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
trainer_args = TrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16,
    fp16=True,
)

In [ ]:
from ast import arg
trainer = Trainer(
    model=model_pegasus, args=trainer_args,
    processing_class=tokenizer, data_collator=seq2seq_data,
    train_dataset=dataset_samsum['test'],
    eval_dataset=dataset_samsum['validation'])

In [ ]:
trainer.train()

In [ ]:
##Evaluation##
def generate_batch_sized_chunks(list_of_elements, batch_size):
  for i in range(0, len(list_of_elements), batch_size):
    yield list_of_elements[i : i + batch_size]

def calculate_metrics_on_test_ds(
    dataset, metric, model, tokenizer,
    batch_size=16, device=device,
    column_text="article",
    column_summary="highlights"
):
  article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
  target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))
  for article_batch, target_batch in tqdm(
      zip(article_batches, target_batches), total=len(article_batches)):

      inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                         padding="max_length", return_tensors="pt")

      summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                  attention_mask=inputs["attention_mask"].to(device),
                                 length_penalty=0.8, num_beams=8, max_length=128)
      #Decode the generated texts, replace the token and add the decoded texts with reference to the metric.
      decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                            clean_up_tokenization_spaces=True)
                           for s in summaries]
      decoded_summaries = [d.replace(" ", " ") for d in decoded_summaries]

      metric.add_batch(predictions=decoded_summaries, references=target_batch)

  #Computing the metric and returning the ROUGE scores.
  score = metric.compute()
  return score

In [ ]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = load_metric('rouge')

In [ ]:
score = calculate_metrics_on_test_ds(
    dataset_samsum['test'][0:15], rouge_metric,
    trainer.model, tokenizer, batch_size = 2,
    column_text = 'dialogue', column_summary= 'summary'
)
rouge_dict = {rn: score[rn] for rn in rouge_names}
pd.DataFrame(rouge_dict, index=[f'pegasus'])

In [ ]:
model_pegasus.save_pretrained("pegasus-samsum-model")

In [ ]:
tokenizer.save_pretrained("tokenizer")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

In [ ]:
##Prediction
generate_kwargs = {
    "length_penalty":0.8,
    "num_beams":8,
    "max_length":100
}

sample_text = dataset_samsum['test'][0]['dialogue']

reference = dataset_samsum['test'][0]['summary']

# Update the model path to the local directory
pipe = pipeline('summarization', model='pegasus-samsum-model', tokenizer='tokenizer') # Changed model path

print('Dialogue:')
print(sample_text)

print("\Reference Summary:")
print(reference)

print("\nModel Summary:")
print(pipe(sample_text, **generate_kwargs)[0]['summary_text'])